# Plant Disease Classifier — Training Notebook (v2)

Improvements over the previous version:

1. **Proper 3-way split (70/15/15 train/val/test)** — the val set is used only for checkpointing/LR scheduling during training; the held-out **test set** (never touched until the end) is what the final classification report / confusion matrix are computed on. This gives you an honest accuracy number.
2. **Reduced augmentation** — dropped `hue` jitter from ColorJitter. Disease symptoms on leaves are largely color-based (chlorosis, browning, yellowing) — hue jitter actively fights the signal you want the model to learn. Kept flip/rotation/mild brightness-contrast-saturation jitter.
3. **Two-phase fine-tuning**:
   - Phase 1: backbone frozen, only the new classifier head trains (fast, stabilizes the head).
   - Phase 2: full network unfrozen, fine-tuned at a much lower LR. This avoids destroying the pretrained ImageNet features early in training.
4. **AdamW + weight decay + label smoothing** for better generalization on classes that look visually similar (e.g. early vs. late stage of the same disease).
5. **Early stopping** on top of `ReduceLROnPlateau`, so training stops once val accuracy plateaus instead of running a fixed epoch count.
6. **Mixed precision (AMP)** — meaningfully faster on a Colab T4/A100 GPU with no accuracy cost.
7. **Self-contained checkpoint** — the saved `.pth` includes `class_names`, `image_size`, normalization `mean`/`std`, and `model_name`, so the separate inference script needs zero hardcoded assumptions about your run.

> **Note on PlantVillage specifically:** its images are lab photos on a plain background. Models routinely hit 99%+ accuracy on it, but that number does **not** reliably transfer to real field/phone photos with messy backgrounds, lighting, and occlusion. Treat the test accuracy below as an upper bound on real-world performance, not a guarantee.

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload your PlantVillage dataset zip (e.g. data.zip)

In [ ]:
import os
os.makedirs("models", exist_ok=True)

In [ ]:
import os
print(os.listdir("/content"))

In [ ]:
import zipfile

zip_path = "/content/data.zip"   # <-- change this if your uploaded zip has a different name
extract_path = "/content/data"

if not os.path.isdir(extract_path):
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_path)
    print("Extraction complete.")
else:
    print("Data already extracted, skipping.")

In [ ]:
# Standard Library
import os
import copy
import json
import random
import time
from pathlib import Path

# Numerical Computing & Data Handling
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# TorchVision
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models

# Data Loading Utilities
from torch.utils.data import DataLoader, random_split, Subset
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

In [ ]:
# ================================
# Reproducibility
# ================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
DATASET_ROOT = Path("/content/data")
IMAGE_SIZE = 224
BATCH_SIZE = 32

# Two-phase training schedule
HEAD_EPOCHS = 5          # phase 1: backbone frozen, train classifier head only
FINE_TUNE_EPOCHS = 25    # phase 2: full network unfrozen (early stopping will likely cut this short)
HEAD_LR = 1e-3
FINE_TUNE_LR = 1e-4      # much lower than head LR to protect pretrained features
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
EARLY_STOP_PATIENCE = 5  # stop phase 2 if val acc doesn't improve for this many epochs

MODEL_NAME = "efficientnet_b0"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

In [ ]:
print(f"Using device: {DEVICE}")
if DEVICE.type == "cpu":
    print("WARNING: No GPU detected. Training will be slow. "
          "Go to Runtime > Change runtime type > GPU in Colab.")

In [ ]:
# ================================
# Auto-detect the actual class-folder directory
# ================================
def find_dataset_dir(root: Path):
    for dirpath, dirnames, filenames in os.walk(root):
        if not dirnames:
            continue
        p = Path(dirpath)
        subdirs = [p / d for d in dirnames]
        if all(any(f.is_file() for f in sd.iterdir()) for sd in subdirs if sd.exists()):
            return p
    raise FileNotFoundError(f"Could not find a class-folder structure under {root}")

DATASET_PATH = find_dataset_dir(DATASET_ROOT)
print(f"Detected dataset path: {DATASET_PATH}")

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
        # hue intentionally omitted — disease symptoms are a color signal, don't jitter it away
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

eval_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

In [ ]:
# ================================
# Build three dataset "views" (same images, different transforms), then
# apply an identical 70/15/15 split to all three via Subset so the
# val/test sets never see training-time augmentation.
# ================================
train_dataset_full = datasets.ImageFolder(root=DATASET_PATH, transform=train_transforms)
eval_dataset_full = datasets.ImageFolder(root=DATASET_PATH, transform=eval_transforms)

class_names = train_dataset_full.classes
print(f"Total Classes : {len(class_names)}")
print(class_names)

assert train_dataset_full.samples == eval_dataset_full.samples, \
    "Dataset orderings differ — cannot safely share split indices."

n_total = len(train_dataset_full)
n_train = int(0.70 * n_total)
n_val = int(0.15 * n_total)
n_test = n_total - n_train - n_val

generator = torch.Generator().manual_seed(SEED)
train_indices, val_indices, test_indices = random_split(
    range(n_total), [n_train, n_val, n_test], generator=generator
)

train_dataset = Subset(train_dataset_full, train_indices.indices)
val_dataset = Subset(eval_dataset_full, val_indices.indices)
test_dataset = Subset(eval_dataset_full, test_indices.indices)

print(f"Training Images   : {len(train_dataset)}")
print(f"Validation Images : {len(val_dataset)}")
print(f"Test Images        : {len(test_dataset)}")

In [ ]:
# ================================
# Class balance check — imbalanced classes can silently hurt minority-class recall
# ================================
from collections import Counter
train_labels = [train_dataset_full.samples[i][1] for i in train_indices.indices]
counts = Counter(train_labels)
print("Min class count:", min(counts.values()), "| Max class count:", max(counts.values()))
plt.figure(figsize=(14, 4))
plt.bar([class_names[i] for i in sorted(counts)], [counts[i] for i in sorted(counts)])
plt.xticks(rotation=90)
plt.ylabel("Training images")
plt.title("Class distribution (training split)")
plt.tight_layout()
plt.show()

In [ ]:
# ================================
# Create DataLoaders
# ================================
NUM_WORKERS = 2

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Training Batches   : {len(train_loader)}")
print(f"Validation Batches : {len(val_loader)}")
print(f"Test Batches        : {len(test_loader)}")

In [ ]:
# ================================
# Check One Batch + Display Sample Images
# ================================
images, labels = next(iter(train_loader))
print("Image Batch Shape :", images.shape)
print("Label Shape :", labels.shape)

def imshow(img):
    img = img.permute(1, 2, 0).numpy()
    mean = np.array(IMAGENET_MEAN)
    std = np.array(IMAGENET_STD)
    img = std * img + mean
    img = np.clip(img, 0, 1)
    plt.imshow(img)
    plt.axis("off")

plt.figure(figsize=(15, 8))
for i in range(min(8, len(images))):
    plt.subplot(2, 4, i + 1)
    imshow(images[i])
    plt.title(class_names[labels[i]], fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ================================
# Build EfficientNet-B0 with a fresh classifier head
# ================================
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, len(class_names))
model = model.to(DEVICE)

def set_backbone_trainable(model, trainable: bool):
    for name, param in model.named_parameters():
        if not name.startswith("classifier"):
            param.requires_grad = trainable

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

In [ ]:
# ================================
# Train / eval loop helpers (shared by both phases)
# ================================
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    running_loss, correct, total = 0.0, 0, 0
    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for images, labels in tqdm(loader, desc="Train" if is_train else "Eval"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            if is_train:
                optimizer.zero_grad()
                with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                    outputs = model(images)
                    loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return running_loss / len(loader), 100 * correct / total

In [ ]:
best_val_acc = 0.0
best_epoch = -1
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

def maybe_save_checkpoint(epoch, val_acc):
    global best_val_acc, best_epoch
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        torch.save({
            "model_state_dict": model.state_dict(),
            "class_names": class_names,
            "image_size": IMAGE_SIZE,
            "num_classes": len(class_names),
            "model_name": MODEL_NAME,
            "mean": IMAGENET_MEAN,
            "std": IMAGENET_STD,
            "best_val_acc": best_val_acc,
        }, "models/best_model.pth")
        print(f"  -> New best model saved (val acc {val_acc:.2f}%)")

In [ ]:
# ================================
# PHASE 1: train classifier head only (backbone frozen)
# ================================
set_backbone_trainable(model, trainable=False)
head_optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=HEAD_LR, weight_decay=WEIGHT_DECAY
)

print(f"Phase 1: training classifier head for {HEAD_EPOCHS} epochs (backbone frozen)")
for epoch in range(1, HEAD_EPOCHS + 1):
    print(f"\n[Head] Epoch [{epoch}/{HEAD_EPOCHS}]")
    print("-" * 50)
    train_loss, train_acc = run_epoch(model, train_loader, optimizer=head_optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, optimizer=None)

    history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc); history["val_acc"].append(val_acc)

    print(f"Train Loss : {train_loss:.4f} | Train Acc : {train_acc:.2f}%")
    print(f"Val Loss   : {val_loss:.4f} | Val Acc   : {val_acc:.2f}%")

    maybe_save_checkpoint(epoch, val_acc)

In [ ]:
# ================================
# PHASE 2: fine-tune the full network at a low LR, with early stopping
# ================================
set_backbone_trainable(model, trainable=True)
fine_tune_optimizer = optim.AdamW(model.parameters(), lr=FINE_TUNE_LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(fine_tune_optimizer, mode="max", factor=0.5, patience=2)

epochs_without_improvement = 0
print(f"\nPhase 2: fine-tuning full network for up to {FINE_TUNE_EPOCHS} epochs "
      f"(early stop patience={EARLY_STOP_PATIENCE})")

for epoch in range(1, FINE_TUNE_EPOCHS + 1):
    global_epoch = HEAD_EPOCHS + epoch
    print(f"\n[Fine-tune] Epoch [{epoch}/{FINE_TUNE_EPOCHS}]")
    print("-" * 50)

    train_loss, train_acc = run_epoch(model, train_loader, optimizer=fine_tune_optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, optimizer=None)
    scheduler.step(val_acc)

    history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc); history["val_acc"].append(val_acc)

    print(f"Train Loss : {train_loss:.4f} | Train Acc : {train_acc:.2f}%")
    print(f"Val Loss   : {val_loss:.4f} | Val Acc   : {val_acc:.2f}%")
    print(f"LR now     : {fine_tune_optimizer.param_groups[0]['lr']:.2e}")

    prev_best = best_val_acc
    maybe_save_checkpoint(global_epoch, val_acc)

    if best_val_acc > prev_best:
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping: no val improvement for {EARLY_STOP_PATIENCE} epochs.")
            break

print(f"\nTraining Completed! Best Validation Accuracy : {best_val_acc:.2f}% (epoch {best_epoch})")

In [ ]:
# ================================
# Plot Training Curves
# ================================
epochs_range = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history["train_loss"], label="Train Loss")
plt.plot(epochs_range, history["val_loss"], label="Val Loss")
plt.axvline(HEAD_EPOCHS + 0.5, color="gray", linestyle="--", label="Head -> Fine-tune")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Loss over Epochs"); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history["train_acc"], label="Train Acc")
plt.plot(epochs_range, history["val_acc"], label="Val Acc")
plt.axvline(HEAD_EPOCHS + 0.5, color="gray", linestyle="--", label="Head -> Fine-tune")
plt.xlabel("Epoch"); plt.ylabel("Accuracy (%)"); plt.title("Accuracy over Epochs"); plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================
# Final Evaluation on the held-out TEST set (never seen during training or checkpointing)
# ================================
checkpoint = torch.load("models/best_model.pth", map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

all_preds, all_labels, all_confidences = [], [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Evaluating on test set"):
        images = images.to(DEVICE)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        conf, predicted = torch.max(probs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_confidences.extend(conf.cpu().numpy())

test_acc = 100 * np.mean(np.array(all_preds) == np.array(all_labels))
print(f"Held-out TEST accuracy: {test_acc:.2f}%  (mean prediction confidence: {np.mean(all_confidences)*100:.2f}%)")
print()
print(classification_report(all_labels, all_preds, target_names=class_names))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=False, cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion Matrix (test set)")
plt.xticks(rotation=90); plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ================================
# Save class names alongside the model (handy for the inference script / any app)
# ================================
with open("models/class_names.json", "w") as f:
    json.dump(class_names, f, indent=2)

print("Saved models/best_model.pth and models/class_names.json")

In [ ]:
from google.colab import files

files.download("models/best_model.pth")
files.download("models/class_names.json")